# ScreamingFace · quickstart

Connect three model providers, combine their models into one Fusion, evaluate it on five real GPQA
Diamond questions, and compare the Fusion with its strongest member.

This is the shortest supported path: **connect → compose → evaluate → compare**. Its core
evaluation path remains **compose → evaluate → compare**. Evaluation uses real model responses
through the configured ScreamingFace engine; it never substitutes an offline result.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

The local AI Gateway starts with an empty provider profile store. The first cell opens the
ScreamingFace provider panel, which stores provider credentials through the engine. If a selected
provider is disconnected, evaluation raises one actionable `ConnectionRequiredError` before model
calls; provider authentication never becomes repeated per-case failures.

### Gemini compatibility · July 2026

Some newly created Google API projects may receive `model no longer available` for Gemini 2.5
even when their quota dashboard displays Gemini 2.5 limits. The local AI Gateway used by this
notebook does not yet register Google's recommended `gemini-3.5-flash` or
`gemini-3.1-pro-preview` replacements. If that happens, replace the Gemini member below with
another connected model advertised by `sf.models.list()`.

Hugging Face does not provide Gemini through this integration. The forthcoming Hugging Face route
is for open models such as DeepSeek and GLM through pinned inference providers; Gemini 3 still
requires explicit AI Gateway support.

GPQA is fetched through this notebook's Hugging Face session, so accept its dataset terms and
authenticate this Python environment when required:

```bash
huggingface-cli login
```

The five-case example makes 15 model calls: three Fusion members for each question. Majority vote,
answer-key grading, and the final comparison make no additional provider calls.

## 1 · Connect

In [ ]:
import screamingface as sf

sf.connect()

Connect each provider used below. The panel sends credentials only to the configured
ScreamingFace engine and shows the engine origin before you act.

## 2 · Compose

In [ ]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini/2.5-flash",
        "claude/sonnet-4.6",
    ],
    prompt="Return only the answer letter: A, B, C, or D.",
    reducer=sf.reducers.MajorityVote(),
)

fusion

Each member answers the same multiple-choice question. `MajorityVote` selects the
most common exact answer and breaks a tie by stable member order. Fusion construction is local and
does not call a model.

## 3 · Evaluate

In [ ]:
report = fusion.evaluate("gpqa@1", first=5)

# Equivalent staged API:
# benchmark = sf.benchmarks.load("gpqa@1")
# run = fusion.run(benchmark, first=5)
# grades = run.grade()
# report = grades.aggregate()

`evaluate(...)` loads the pinned GPQA Diamond definition through this process,
executes the three-member Fusion for the first five canonical cases, checks the answers against the
sealed answer key, and returns one paired comparison. Missing work remains an explicit failure; it
is never silently scored as zero. In a notebook, one compact live panel shows requirement checks,
case execution, grading, and aggregation before giving way to the final report. Pass
`progress=False` to hide it, or `progress=True` to force the same progress outside notebooks.

## 4 · Compare

In [ ]:
report

Read `gain` first:

- `score` is the Fusion's accuracy across the successfully paired cases;
- `baseline` is the best individual member's accuracy on those same cases; and
- `gain` is `score - baseline`.

A positive gain means the combination outperformed every member on the evaluated cases. A strong
score with zero gain means the Fusion matched, but did not improve on, its strongest member.